# Speculative Decoding Lab (Colab)

이 노트북은 **Colab GPU**에서 speculative decoding을 실습하기 위한 예제입니다.
- Toy 데모 실행
- 공개 모델(Qwen 2.5 7B/0.5B)로 speculative decoding 실행


## 0) 런타임 설정
Colab에서 `런타임 > 런타임 유형 변경 > GPU`로 설정하세요.


In [ ]:
!nvidia-smi

## 1) 코드/의존성 준비


In [ ]:
# 필요 시 본인 레포 URL로 교체
!rm -rf /content/jinsunghub
%cd /content
!git clone <YOUR_REPO_URL> jinsunghub
%cd /content/jinsunghub/speculative-decoding-lab
!pip install -U pip
!pip install -r requirements.txt
!python spec_decode_hf.py --help
!python - <<'PY'
from pathlib import Path
text = Path('spec_decode_hf.py').read_text(encoding='utf-8')
print('VERSION_OK' if 'builtin-assistant-v2' in text else 'OLD_SCRIPT')
PY


## 2) Hugging Face 로그인 (선택)
게이트 모델(Llama 2 등)을 쓸 때 필요합니다.


In [ ]:
from huggingface_hub import login
login()

## 3) Toy 데모 실행


In [ ]:
!python spec_decode_toy.py --prompt "I" --max-new-tokens 20 --k 4 --seed 7

## 4) 7B(target) + 0.5B(draft) speculative decoding (공개 모델)


In [ ]:
!python spec_decode_hf.py --target-model Qwen/Qwen2.5-7B-Instruct --draft-model Qwen/Qwen2.5-0.5B-Instruct --prompt "Explain speculative decoding in simple Korean." --max-new-tokens 64 --k 8 --skip-baseline

### Llama 2 403 오류가 뜨는 이유
`meta-llama/Llama-2-7b-hf`는 gated repo입니다. 접근 승인 없는 계정은 403이 납니다.
Llama 2를 꼭 쓰려면 HF 모델 페이지에서 access 승인 후 `login()`한 계정으로 실행하세요.


## 5) 실험 팁
- 먼저 `--skip-baseline`으로 speculative만 확인 후, 필요하면 baseline 비교
- `--num-assistant-tokens`를 4/8/12로 바꿔서 속도 비교
- OOM/지연 시 `--max-new-tokens`를 32~64로 축소
